In [1]:
import pandas as pd
import numpy as np
import os
import gc
import random
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. 設定 (Configuration)
# ==============================================================================
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'
else:
    INPUT_DIR = '.' 

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_ventilator_lstm.csv'

# ハイパーパラメータ
MAX_LEN = 40
BATCH_SIZE = 64
N_FOLDS = 5
EPOCHS = 35           # LSTMは収束まで少し回数が必要です
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5   # 特徴量が多いので正則化は弱めに

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# ==============================================================================
# 2. 特徴量エンジニアリング (PID Features)
# ==============================================================================
def add_pid_features(df):
    """
    Google Brain Ventilatorコンペの上位解法を模倣。
    時系列の「微分(Diff)」「積分(Cumsum)」「ラグ(Lag)」を追加します。
    """
    df_eng = df.copy()
    
    # 除外カラム
    ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
    base_cols = [c for c in df_eng.columns if c not in ignore_cols]
    
    print(f"Generating PID features for {len(base_cols)} base columns...")
    
    # Groupbyオブジェクトを先に作成（高速化）
    grouped = df_eng.groupby('sample_id')
    
    new_cols = []
    
    for col in base_cols:
        # 1. 時間重み付け (Time-Weighting: あなたのアイデア)
        # 後半のデータを重視する「学習効果」ゲイン
        max_day = df_eng['day_n'].max()
        gain = 1.0 + (df_eng['day_n'] / max_day * 0.5)
        
        # 基本特徴量 (Weighted)
        w_col_name = f'{col}_weighted'
        df_eng[w_col_name] = df_eng[col] * gain
        # PID生成のベースはこの重み付きデータを使います
        
        # --- D (微分): 変化の勢い ---
        # 1つ前の時刻との差 (速度)
        df_eng[f'{col}_diff1'] = grouped[col].diff().fillna(0) * gain
        # 2つ前の時刻との差 (加速度的な要素)
        df_eng[f'{col}_diff2'] = grouped[col].diff(2).fillna(0) * gain
        
        # --- I (積分): 蓄積 ---
        # そのサンプルの開始時点からの累積和 (疲労や活性化の蓄積)
        # ※値が大きくなりすぎるのを防ぐため、スケーリング前に平均で割る等の処理も有効ですが
        #   今回はRobustScalerに任せます。
        df_eng[f'{col}_cumsum'] = grouped[col].cumsum()
        
        # --- Lag (過去の状態) ---
        # Ventilatorコンペでは「R」「C」という固定値との相互作用が効きましたが、
        # ここでは「直前の値」を明示的に入れます。
        df_eng[f'{col}_lag1'] = grouped[col].shift(1).fillna(0)
        
    return df_eng

print(">>> Loading Data & Engineering Features...")
train = pd.read_csv(TRAIN_PATH).fillna(0)
test = pd.read_csv(TEST_PATH).fillna(0)

# 特徴量生成 (少し時間がかかります)
train = add_pid_features(train)
test = add_pid_features(test)

# 特徴量カラムの選定
target_col = 'lever'
ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
# 生成された全カラムを取得
train_feats = [c for c in train.columns if c not in ignore_cols]
test_feats = set(test.columns)
# Train/Test共通のカラムのみ使用
feature_cols = [c for c in train_feats if c in test_feats]

print(f"Total Input Features: {len(feature_cols)}")

# スケーリング
# 累積和(cumsum)などは外れ値が出やすいので、StandardScalerよりRobustScalerが安全です
scaler = RobustScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

# ==============================================================================
# 3. Dataset
# ==============================================================================
class BrainDataset(Dataset):
    def __init__(self, df, feature_cols, target_col=None, max_len=40):
        self.df = df
        self.sample_ids = df['sample_id'].unique()
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.max_len = max_len
        self.grouped = df.groupby('sample_id')

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        group = self.grouped.get_group(sample_id)
        
        x = group[self.feature_cols].values
        seq_len = len(x)
        
        x_pad = np.zeros((self.max_len, len(self.feature_cols)), dtype=np.float32)
        x_pad[:seq_len, :] = x
        mask = np.zeros((self.max_len,), dtype=np.float32)
        mask[:seq_len] = 1.0
        
        ret = {'x': torch.tensor(x_pad), 'mask': torch.tensor(mask), 'id': sample_id}
        if self.target_col:
            y = group[self.target_col].values
            y_pad = np.zeros((self.max_len,), dtype=np.float32)
            y_pad[:seq_len] = y
            ret['y'] = torch.tensor(y_pad)
        return ret

# ==============================================================================
# 4. Model: Deep Bi-LSTM (Ventilator Style)
# ==============================================================================
class VentilatorLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_layers=4):
        super().__init__()
        
        # 特徴量を高次元空間へ射影
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        
        # Deep Bi-LSTM
        # Ventilatorコンペでは4層〜5層が主流でした
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.1 # 層間のドロップアウト
        )
        
        # Regression Head
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2), # Bi-Directionalなので2倍
            nn.Linear(hidden_dim * 2, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )
        
        # 重み初期化 (LSTMの学習安定化のため)
        self._init_weights()

    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                param.data.fill_(0)

    def forward(self, x):
        # x: [Batch, Time, Features]
        x = self.embedding(x)
        
        # LSTM output: [Batch, Time, Hidden*2]
        x, _ = self.lstm(x)
        
        # Regression
        out = self.head(x)
        return out.squeeze(-1)

# ==============================================================================
# 5. Training Loop
# ==============================================================================
final_test_preds = {} 
test_counts = {}

unique_samples = train['sample_id'].unique()
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
criterion = nn.MSELoss(reduction='none')

print(f"\n>>> Starting {N_FOLDS}-Fold Cross Validation (Deep Bi-LSTM + PID)...")

# Test Dataset
test_ds = BrainDataset(test, feature_cols, None, MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

for fold, (train_idx, val_idx) in enumerate(kf.split(unique_samples)):
    print(f"\n=== Fold {fold+1}/{N_FOLDS} ===")
    
    train_ids, val_ids = unique_samples[train_idx], unique_samples[val_idx]
    train_ds = BrainDataset(train[train['sample_id'].isin(train_ids)], feature_cols, target_col, MAX_LEN)
    val_ds = BrainDataset(train[train['sample_id'].isin(val_ids)], feature_cols, target_col, MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデル初期化 (入力次元が増えているので注意)
    model = VentilatorLSTM(
        input_dim=len(feature_cols),
        hidden_dim=256,
        num_layers=4
    ).to(DEVICE)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    best_loss = float('inf')
    best_model_path = f"best_lstm_pid_fold{fold}.pth"
    
    # --- 学習 ---
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in train_loader:
            x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
            optimizer.zero_grad()
            preds = model(x)
            loss = (criterion(preds, y) * m).sum() / m.sum()
            loss.backward()
            
            # LSTMは勾配爆発しやすいのでクリッピング推奨
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_loss += loss.item()
        
        scheduler.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                x, y, m = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['mask'].to(DEVICE)
                preds = model(x)
                loss = (criterion(preds, y) * m).sum() / m.sum()
                val_loss += loss.item()
        
        avg_val = val_loss / len(val_loader)
        
        if avg_val < best_loss:
            best_loss = avg_val
            torch.save(model.state_dict(), best_model_path)
            
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"  Ep {epoch+1:2d} | Train: {train_loss/len(train_loader):.4f} | Val: {avg_val:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    print(f"  >> Best Val: {best_loss:.5f}")

    # --- 推論 & 中間保存 ---
    print("  Predicting Test Data...")
    try:
        model.load_state_dict(torch.load(best_model_path))
        model.eval()
        with torch.no_grad():
            for batch in test_loader:
                x = batch['x'].to(DEVICE)
                mask = batch['mask'].to(DEVICE)
                s_ids = batch['id']
                preds = model(x).cpu().numpy()
                mask = mask.cpu().numpy()
                for i, s_id in enumerate(s_ids):
                    v_len = int(mask[i].sum())
                    for t, val in enumerate(preds[i, :v_len]):
                        k = f"{s_id}_{t}"
                        final_test_preds[k] = final_test_preds.get(k, 0) + val
                        test_counts[k] = test_counts.get(k, 0) + 1
                        
        # Foldごとに途中結果を保存
        temp_results = [{'id': k, 'lever': max(0, v / test_counts[k])} for k, v in final_test_preds.items()]
        pd.DataFrame(temp_results).to_csv(f'temp_submission_fold{fold+1}.csv', index=False)
        print(f"  -> Intermediate result saved: temp_submission_fold{fold+1}.csv")
        
    except Exception as e:
        print(f"  ERROR in Inference Fold {fold+1}: {e}")

    # メモリ解放
    del model, optimizer, scheduler, train_ds, val_ds, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

# ==============================================================================
# 6. Final Submission
# ==============================================================================
print("\n>>> Creating Final LSTM+PID Submission...")
if len(final_test_preds) > 0:
    results = [{'id': k, 'lever': max(0, v / test_counts[k])} for k, v in final_test_preds.items()]
    df_lstm = pd.DataFrame(results)
    df_lstm.to_csv(SUBMISSION_PATH, index=False)
    print(f"SUCCESS: {SUBMISSION_PATH} created.")
else:
    print("ERROR: No predictions generated.")

Using Device: cuda
>>> Loading Data & Engineering Features...
Generating PID features for 44 base columns...
Generating PID features for 44 base columns...
Total Input Features: 264

>>> Starting 5-Fold Cross Validation (Deep Bi-LSTM + PID)...

=== Fold 1/5 ===
  Ep  1 | Train: 1.2820 | Val: 1.1904 | LR: 0.000998
  Ep  5 | Train: 0.8906 | Val: 1.0930 | LR: 0.000951
  Ep 10 | Train: 0.7426 | Val: 0.9480 | LR: 0.000812
  Ep 15 | Train: 0.5803 | Val: 0.9509 | LR: 0.000612
  Ep 20 | Train: 0.3484 | Val: 1.0094 | LR: 0.000389
  Ep 25 | Train: 0.1910 | Val: 1.0378 | LR: 0.000189
  Ep 30 | Train: 0.1145 | Val: 1.0544 | LR: 0.000050
  Ep 35 | Train: 0.0939 | Val: 1.0607 | LR: 0.000001
  >> Best Val: 0.91646
  Predicting Test Data...
  -> Intermediate result saved: temp_submission_fold1.csv

=== Fold 2/5 ===
  Ep  1 | Train: 1.4305 | Val: 1.0813 | LR: 0.000998
  Ep  5 | Train: 0.9498 | Val: 0.9869 | LR: 0.000951


KeyboardInterrupt: 